# Follow one MRI batch through the pipeline

Submit the baseline cluster job from `README.md` first. While it runs, use this notebook to follow the same four pieces imported by `main.py`: data loading, the model, the loss, and the optimizer. Run cells from top to bottom from the repository root.

## Part 1 — data loading

The local sample has one four-channel MRI and one raw mask per patient. The filenames match across the two folders:

```text
../data/Task01_BrainTumour/  # outside this repository
├── dataset.json
├── imagesTr/BRATS_001.nii.gz   # FLAIR, T1, T1gd, T2
└── labelsTr/BRATS_001.nii.gz   # integer tumor labels
```

Use `glob` to choose a patient. Change `[0]` to inspect another one.


In [ ]:
from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
import torch
from torch.nn import functional as F
from monai.data import DataLoader, Dataset
from monai.transforms import (
    Compose, EnsureChannelFirstd, LoadImaged, Orientationd,
    RandCropByPosNegLabeld, ScaleIntensityd, ToTensord,
)

DATA_ROOT = Path(os.environ.get("DATA_ROOT", "../data/Task01_BrainTumour"))
image_path = sorted((DATA_ROOT / "imagesTr").glob("*.nii.gz"))[0]
files = {
    "image": str(image_path),
    "label": str(DATA_ROOT / "labelsTr" / image_path.name),
    "case_id": image_path.name.removesuffix(".nii.gz"),
}
channel_names = json.loads((DATA_ROOT / "dataset.json").read_text())["modality"]
print("Patient:", files["case_id"])
print("Image:", files["image"])
print("Mask:", files["label"])


`basic_train_transforms` below is copied from `src/data_loader.py`. MONAI reads the image and mask, puts channels first, orients them together, scales each MRI channel, and samples patches around foreground and background voxels. The label stays a one-channel map of class numbers: `0` background, `1` edema, `2` non-enhancing tumor, `3` enhancing tumor. There is no label conversion.


In [ ]:
def basic_train_transforms(patch_size, patches_per_case):
    """Load images and masks, then ask MONAI to sample training patches."""
    steps = [
        LoadImaged(keys=["image", "label"]),
        EnsureChannelFirstd(keys=["image", "label"]),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        ScaleIntensityd(keys="image", channel_wise=True),
        RandCropByPosNegLabeld(
            keys=["image", "label"], label_key="label", spatial_size=patch_size,
            pos=1, neg=1, num_samples=patches_per_case,
        ),
    ]
    steps.append(ToTensord(keys="image", dtype=torch.float32))
    steps.append(ToTensord(keys="label", dtype=torch.int64))
    return Compose(steps)


Validation needs the whole MRI. `val_transforms` uses the same loading steps but leaves out the random crop. `main.py` later uses sliding-window inference on this full MRI.


In [ ]:
def val_transforms():
    """Load the full image for sliding-window validation; do not crop."""
    steps = [
        LoadImaged(keys=["image", "label"]),
        EnsureChannelFirstd(keys=["image", "label"]),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        ScaleIntensityd(keys="image", channel_wise=True),
    ]
    steps.append(ToTensord(keys="image", dtype=torch.float32))
    steps.append(ToTensord(keys="label", dtype=torch.int64))
    return Compose(steps)


MONAI's `Dataset` and `DataLoader` turn those transforms into batches. The training batch contains one small patch; the validation batch contains the full patient. `main.py` uses `CacheDataset` with the same transforms to avoid repeatedly loading one patient.


In [ ]:
train_loader = DataLoader(
    Dataset([files], transform=basic_train_transforms((32, 32, 32), 1)),
    batch_size=1,
)
val_loader = DataLoader(Dataset([files], transform=val_transforms()), batch_size=1)
train_batch = next(iter(train_loader))
val_batch = next(iter(val_loader))
print("Training image:", tuple(train_batch["image"].shape))
print("Training target:", tuple(train_batch["label"].shape))
print("Validation image:", tuple(val_batch["image"].shape))
print("Classes in this training patch:", torch.unique(train_batch["label"]).tolist())


Inspect one MRI slice from `val_loader`. The orange overlay marks tumor (any label greater than zero). Change `slice_index` and `channel_index` in the next cell, then rerun it to inspect another slice or MRI contrast. This plot works without JupyterLab widget extensions.


In [ ]:
mri = val_batch["image"][0]
mask = val_batch["label"][0, 0]

def show_slice(z, channel):
    image_slice = torch.rot90(mri[channel, :, :, z].cpu(), 1, (0, 1))
    mask_slice = torch.rot90((mask[:, :, z] > 0).cpu(), 1, (0, 1))
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(image_slice, cmap="gray")
    ax.imshow(mask_slice, cmap="autumn", vmin=0, vmax=1, alpha=mask_slice * 0.45)
    ax.set_title(f"{files['case_id']} — {channel_names[str(channel)]}, slice {z}")
    ax.axis("off")
    plt.show()
    plt.close(fig)

start_slice = int(torch.argmax((mask > 0).sum(dim=(0, 1))))
slice_index = start_slice  # Choose a number from 0 to mri.shape[-1] - 1.
channel_index = 2  # 0=FLAIR, 1=T1, 2=T1gd, 3=T2.
show_slice(slice_index, channel_index)


The image has four MRI channels. The mask has **one** channel and each voxel contains one class number. The model will output four scores per voxel, one for each possible class.

## Part 2 — model architecture

`main.py` imports `BasicUNet` from `src/model.py`. This starter U-Net has **three resolution levels**. A 3D convolution with `kernel_size=3, padding=1` keeps depth, height, and width unchanged. Pooling then halves those spatial dimensions. The decoder upsamples twice, each time joining features from the matching encoder level. The final layer produces four numbers per voxel, for background and the three tumor classes. These are **logits**, not probabilities.

In this notebook, `base_channels=8`: the input has four MRI channels, and encoder level 1 has eight feature channels. This makes the channel change visible even though the spatial size stays the same. Every row in the table below has batch size `1`.


In [ ]:
from src.model import BasicUNet

model = BasicUNet(base_channels=8)  # Small enough to explore on a CPU.
image = train_batch["image"][:1]
target = train_batch["label"][:1]

with torch.no_grad():
    level_1 = model.encoder1(image)
    level_2 = model.encoder2(F.max_pool3d(level_1, 2))
    bottleneck = model.bottleneck(F.max_pool3d(level_2, 2))
    up_2 = F.interpolate(bottleneck, size=level_2.shape[2:], mode="trilinear", align_corners=False)
    decoder_2 = model.decoder2(torch.cat([up_2, level_2], dim=1))
    up_1 = F.interpolate(decoder_2, size=level_1.shape[2:], mode="trilinear", align_corners=False)
    decoder_1 = model.decoder1(torch.cat([up_1, level_1], dim=1))
    logits = model.head(decoder_1)

print("One convolution block:", model.encoder1)
print(f"{'Step':<20} {'Channels':>8} {'Depth × height × width':>24}")
for name, tensor in [
    ("Input", image),
    ("Encoder level 1", level_1),
    ("Encoder level 2", level_2),
    ("Bottleneck", bottleneck),
    ("Upsample to level 2", up_2),
    ("Decoder level 2", decoder_2),
    ("Upsample to level 1", up_1),
    ("Decoder level 1", decoder_1),
    ("Output logits", logits),
    ("Target mask", target),
]:
    print(f"{name:<20} {tensor.shape[1]:>8} {str(tuple(tensor.shape[2:])):>24}")
print("Trainable parameters:", sum(p.numel() for p in model.parameters()))


## Part 3 — loss and optimizer

`main.py` imports `segmentation_loss` from `src/loss.py` and `build_optimizer` from `src/optim.py`. Cross entropy compares the four output scores with the single class number in the target mask. The optimizer uses the loss gradient to update model weights. Below is **one training step**; `main.py` repeats it for every batch and epoch.


In [ ]:
from src.loss import segmentation_loss
from src.optim import build_optimizer

optimizer, scheduler = build_optimizer(model.parameters(), learning_rate=1e-3, epochs=1)
optimizer.zero_grad()
logits = model(image)
loss = segmentation_loss(logits, target)
loss.backward()
optimizer.step()

print("Loss:", float(loss.item()))
print("Optimizer:", type(optimizer).__name__)
print("Learning rate:", optimizer.param_groups[0]["lr"])
print("Scheduler:", type(scheduler).__name__ if scheduler else "none")


## Now follow the cluster run

You have seen the same parts that `main.py` connects: a MONAI batch, U-Net logits, the loss, and an optimizer update. Check the baseline job's log and W&B run. Note the first training loss and the first validation Dice for edema, non-enhancing tumor, and enhancing tumor. Those are the reference for your later code changes.
